<a href="https://colab.research.google.com/github/MehediHasanUthan/ml-journey/blob/main/projects/day01-cifar10-baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **#Neural Network Definition**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
#Dataset download and loading
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True,  download=True, transform=transform)
test_dataset  = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False, num_workers=2)

#Neural Network
class CIFAR10_CNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(3,   32,  3, padding=1)
        self.conv2 = nn.Conv2d(32,  64,  3, padding=1)
        self.conv3 = nn.Conv2d(64,  128, 3, padding=1)
        self.pool  = nn.MaxPool2d(2, 2)
        self.fc1   = nn.Linear(128 * 4 * 4, 256)
        self.fc2   = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))   # 32 → 16
        x = self.pool(F.relu(self.conv2(x)))   # 16 → 8
        x = self.pool(F.relu(self.conv3(x)))   # 8  → 4
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

#Model Definition
model = CIFAR10_CNN(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

#Training Functions Definition
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)
    model.train()
    return 100. * correct / total

def train(model, train_loader, test_loader, epochs=10):
    for epoch in range(epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)
            pass

        train_acc = 100. * correct / total
        test_acc  = evaluate(model, test_loader)
        avg_loss  = running_loss / len(train_loader)
        print(f"Epoch [{epoch+1}/{epochs}]  Loss: {avg_loss:.3f}  Train Acc: {train_acc:.2f}%  Test Acc: {test_acc:.2f}%")

#training
train(model, train_loader, test_loader, epochs=10)

100%|██████████| 170M/170M [00:04<00:00, 41.7MB/s]


Epoch [1/10]  Loss: 1.391  Train Acc: 49.39%  Test Acc: 62.08%
Epoch [2/10]  Loss: 0.940  Train Acc: 66.70%  Test Acc: 66.70%
Epoch [3/10]  Loss: 0.756  Train Acc: 73.69%  Test Acc: 71.95%
Epoch [4/10]  Loss: 0.624  Train Acc: 78.41%  Test Acc: 75.22%
Epoch [5/10]  Loss: 0.523  Train Acc: 81.52%  Test Acc: 75.27%
Epoch [6/10]  Loss: 0.434  Train Acc: 84.67%  Test Acc: 74.98%
Epoch [7/10]  Loss: 0.348  Train Acc: 87.76%  Test Acc: 75.44%
Epoch [8/10]  Loss: 0.270  Train Acc: 90.34%  Test Acc: 75.78%
Epoch [9/10]  Loss: 0.209  Train Acc: 92.68%  Test Acc: 74.60%
Epoch [10/10]  Loss: 0.159  Train Acc: 94.46%  Test Acc: 75.28%
